In [5]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# <--- MODIFICATION START: 引入hd95和可能的警告 ---
from medpy.metric.binary import hd95
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning) # MedPy在空掩码时可能产生RuntimeWarning
# <--- MODIFICATION END ---

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ==============================================================================
# --- 1. 配置 (请确认这里的路径) ---
# ==============================================================================
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/busi_bLoss_adapter_run18/checkpoints/checkpoint.pt"
HYDRA_OVERRIDES = [
    "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
    "+model.image_encoder.trunk.use_adapter=True",
    "+model.use_adapter=True",
]
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/Kvasir-SEG_for_SAM2"
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets", "val.txt")

# ==============================================================================
# --- 2. 初始化SAM2模型和预测器 (无需修改) ---
# ==============================================================================
print("Loading SAM2 model in Adapter mode...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam2(
    SAM2_CFG_NAME,
    SAM2_CHECKPOINT_PATH,
    device=device,
    hydra_overrides_extra=HYDRA_OVERRIDES
)
predictor = SAM2ImagePredictor(model)
print(f"Model and Predictor loaded on {device}.")

# ==============================================================================
# --- 3. 定义指标计算函数 (MODIFIED) ---
# ==============================================================================
# <--- MODIFICATION START: 更新指标计算函数以包含HD95 ---
def calculate_metrics(gt_mask, pred_mask):
    """计算Dice, IoU, 和 95% Hausdorff Distance"""
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0

    # Dice 和 IoU 计算 (不变)
    intersection = np.logical_and(gt_mask_bool, pred_mask_bool).sum()
    dice = (2. * intersection) / (gt_mask_bool.sum() + pred_mask_bool.sum() + 1e-8)
    union = gt_mask_bool.sum() + pred_mask_bool.sum() - intersection
    iou = intersection / (union + 1e-8)

    # HD95 计算
    # 豪斯多夫距离在其中一个掩码为空时无定义，此时返回NaN
    if pred_mask_bool.sum() == 0 or gt_mask_bool.sum() == 0:
        hd95_score = np.nan
    else:
        # MedPy的hd95函数需要(预测结果, 金标准)作为输入
        hd95_score = hd95(pred_mask_bool, gt_mask_bool)

    return dice, iou, hd95_score
# <--- MODIFICATION END ---

# ==============================================================================
# --- 4. 遍历测试集、执行分割并评估 (MODIFIED) ---
# ==============================================================================
if not os.path.exists(TEST_SET_FILE):
    raise FileNotFoundError(f"测试集定义文件未找到: {TEST_SET_FILE}")

with open(TEST_SET_FILE, 'r') as f:
    test_sample_names = [line.strip() for line in f.readlines()]

print(f"\nFound {len(test_sample_names)} samples in the test set. Starting evaluation...")

results = []
images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

for sample_name in tqdm(test_sample_names, desc="Evaluating Test Set"):
    image_path = os.path.join(images_dir, sample_name, "00000.jpg")
    mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
    
    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"Warning: Files for sample '{sample_name}' not found. Skipping.")
        continue

    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    predictor.set_image(image_rgb)
    contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print(f"Warning: No lesion found in mask for {sample_name}. Skipping.")
        continue
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    box_prompt = np.array([[x, y, x + w, y + h]])

    masks, scores, logits = predictor.predict(
        box=box_prompt,
        multimask_output=False
    )
    
    pred_mask = (masks[0] * 255).astype(np.uint8)
    
    # <--- MODIFICATION START: 调用新的评估函数 ---
    dice, iou, hd95_score = calculate_metrics(gt_mask, pred_mask)
    # <--- MODIFICATION END ---

    # <--- MODIFICATION START: 记录结果时加入HD95 ---
    category = 'benign' if 'benign' in sample_name else 'malignant'
    results.append({
        "category": category,
        "sample_name": sample_name,
        "dice": dice,
        "iou": iou,
        "hd95": hd95_score # <-- 新增
    })
    # <--- MODIFICATION END ---
    
    predictor.reset_predictor()

# ==============================================================================
# --- 5. 打印最终的总结报告 (MODIFIED) ---
# ==============================================================================
if results:
    df = pd.DataFrame(results)
    
    # <--- MODIFICATION START: 更新聚合函数以包含HD95 ---
    # 按类别计算平均值
    agg_funcs = {
        'dice': ['mean', 'std'],
        'iou': ['mean', 'std'],
        'hd95': ['mean', 'std'],
        'sample_name': ['count']
    }
    category_summary = df.groupby('category').agg(agg_funcs).reset_index()
    
    # 扁平化多级列名
    category_summary.columns = ['_'.join(col).strip() for col in category_summary.columns.values]
    category_summary.rename(columns={'category_': 'Category', 'sample_name_count': 'Count'}, inplace=True)
    # <--- MODIFICATION END ---

    # <--- NEW START: 计算并添加总体平均指标 ---
    overall_metrics = {
        'Category': 'Overall',
        'dice_mean': df['dice'].mean(),
        'dice_std': df['dice'].std(),
        'iou_mean': df['iou'].mean(),
        'iou_std': df['iou'].std(),
        'hd95_mean': df['hd95'].mean(), # .mean() 会自动忽略NaN
        'hd95_std': df['hd95'].std(),
        'Count': len(df)
    }
    
    # 将总体指标作为新行添加到汇总DataFrame中
    overall_df = pd.DataFrame([overall_metrics])
    final_summary = pd.concat([category_summary, overall_df], ignore_index=True)
    # <--- NEW END ---

    print("\n" + "="*80)
    print("--- Evaluation Summary on Test Set (val.txt) ---")
    print("="*80)
    # 打印最终的、包含总体的报告
    print(final_summary.to_string(index=False))
    print("="*80)

else:
    print("No images were processed. Please check paths and file structure.")

print("\n--- Evaluation on the test set is complete! ---")

Loading SAM2 model in Adapter mode...
Model and Predictor loaded on cuda.

Found 200 samples in the test set. Starting evaluation...


Evaluating Test Set: 100%|██████████| 200/200 [00:28<00:00,  6.92it/s]


--- Evaluation Summary on Test Set (val.txt) ---
 Category  dice_mean  dice_std  iou_mean  iou_std  hd95_mean  hd95_std  Count
malignant   0.927612  0.063034  0.870388 0.092639  26.182214  42.56364    200
  Overall   0.927612  0.063034  0.870388 0.092639  26.182214  42.56364    200

--- Evaluation on the test set is complete! ---
